In [21]:
import pandas as pd
import os

# Read the CSV file
input_file = '/mnt/d/heatpump_data/u_values/germany_u_values_cleaned.csv'
df = pd.read_csv(input_file)

print(f"Total rows in original file: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

Total rows in original file: 155
Columns: ['Code_Construction', 'Code_StatusDataset', 'Code_Country', 'Code_ElementType', 'Code_DataType_Construction', 'Code_Construction_ConstructionYearClass', 'Number_Construction_Variant', 'Type_Construction', 'Type_Construction_National', 'Description_Construction', 'Description_Construction_National', 'Remark_Construction', 'Year1_Construction', 'Year2_Construction', 'U']


,Code_Construction,Code_StatusDataset,Code_Country,Code_ElementType,Code_DataType_Construction,Code_Construction_ConstructionYearClass,Number_Construction_Variant,Type_Construction,Type_Construction_National,Description_Construction,Description_Construction_National,Remark_Construction,Year1_Construction,Year2_Construction,U
0,DE.Ceiling.ReEx.01.01,Typology,DE,Ceiling,ReEx,DE.01,1,wooden beam ceiling with visible beams,Holzbalkendecke mit sichtbaren Balken,"wooden beams, cavity filled with clay/straw","Holzbalken, Strohlehmwickel im Gefach",NaN,0,1918.0,1.0
1,DE.Ceiling.ReEx.03.01,Typology,DE,Ceiling,ReEx,DE.03,1,wooden beam ceiling,Holzbalkendecke,"wooden beams, cavity filled with clay, sand or...","Holzbalken, Blindboden, im Gefach: Lehmschlag,...",NaN,0,1968.0,0.8
2,DE.Ceiling.ReEx.04.01,Typology,DE,Ceiling,ReEx,DE.04,1,cavity blocks ceiling,"Rippendecke, Stahlsteindecke, Gitterträgerdecke","cavity elements, reinforcement, concrete filli...","Stahlstein- oder Gitterträgerdecke, Bewehrung,...",NaN,1919,1968.0,2.1
3,DE.Ceiling.ReEx.04.02,Typology,DE,Ceiling,ReEx,DE.04,2,concrete ceiling,Betondecke,"reinforced concrete, 1 cm insulation, cement s...","Stahlbeton, 1 cm Dämmung, Zementestrich",NaN,1949,1968.0,1.6
4,DE.Ceiling.ReEx.06.01,Typology,DE,Ceiling,ReEx,DE.06,1,concrete ceiling with 5 cm insulation,Betondecke mit 5 cm Dämmung,"reinforced concrete, 5 cm insulation, cement s...","Stahlbeton, oberseitig 5 cm Dämmung, Zementest...",NaN,1958,1978.0,0.6


In [22]:
# Parse Code_Construction to extract Country, Element, Renovation, Year_range
# Format: Country.Element.Renovation.Year_range.variant
def parse_code_construction(code):
    """Parse Code_Construction to extract grouping fields.
    
    Args:
        code: Code_Construction string in format Country.Element.Renovation.Year_range.variant
        
    Returns:
        tuple: (Country, Element, Renovation, Year_range)
    """
    parts = code.split('.')
    if len(parts) >= 4:
        return (parts[0], parts[1], parts[2], parts[3])
    return (None, None, None, None)

# Create grouping columns
df[['Country', 'Element', 'Renovation', 'Year_range']] = df['Code_Construction'].apply(
    lambda x: pd.Series(parse_code_construction(x))
)

# Verify parsing
print("Sample parsed data:")
print(df[['Code_Construction', 'Country', 'Element', 'Renovation', 'Year_range', 'Number_Construction_Variant', 'U']].head(10))

Sample parsed data:
       Code_Construction Country  Element Renovation Year_range  \
0  DE.Ceiling.ReEx.01.01      DE  Ceiling       ReEx         01   
1  DE.Ceiling.ReEx.03.01      DE  Ceiling       ReEx         03   
2  DE.Ceiling.ReEx.04.01      DE  Ceiling       ReEx         04   
3  DE.Ceiling.ReEx.04.02      DE  Ceiling       ReEx         04   
4  DE.Ceiling.ReEx.06.01      DE  Ceiling       ReEx         06   
5  DE.Ceiling.ReEx.07.01      DE  Ceiling       ReEx         07   
6  DE.Ceiling.ReEx.07.02      DE  Ceiling       ReEx         07   
7  DE.Ceiling.ReEx.08.01      DE  Ceiling       ReEx         08   
8  DE.Ceiling.ReEx.08.02      DE  Ceiling       ReEx         08   
9  DE.Ceiling.ReEx.09.01      DE  Ceiling       ReEx         09   

   Number_Construction_Variant     U  
0                            1  1.00  
1                            1  0.80  
2                            1  2.10  
3                            2  1.60  
4                            1  0.60  
5       

In [ ]:
# Group by Country, Element, Renovation, Year_range and keep only ONE row with highest U value
# If multiple variants have the same max U value, keep the one with lowest variant number

# Sort the dataframe first: by grouping columns, then by U descending, then by variant ascending
df_sorted = df.sort_values(['Country', 'Element', 'Renovation', 'Year_range', 'U', 'Number_Construction_Variant'],
                           ascending=[True, True, True, True, False, True])

# Use drop_duplicates to keep only the first row for each group (which will be the highest U, lowest variant)
filtered_df = df_sorted.drop_duplicates(subset=['Country', 'Element', 'Renovation', 'Year_range'], 
                                       keep='first').reset_index(drop=True)

# Remove the temporary grouping columns before saving
filtered_df = filtered_df.drop(columns=['Country', 'Element', 'Renovation', 'Year_range'])

print(f"Original rows: {len(df)}")
print(f"Filtered rows: {len(filtered_df)}")
print(f"Rows removed: {len(df) - len(filtered_df)}")

Original rows: 155
Filtered rows: 62
Rows removed: 93


In [ ]:
# Verify the filtering worked correctly - ensure only one variant per group
print("\nVerification - Checking that each group has exactly one variant:")

# Re-create grouping columns for filtered_df for verification
filtered_df_temp = filtered_df.copy()
filtered_df_temp[['Country', 'Element', 'Renovation', 'Year_range']] = filtered_df_temp['Code_Construction'].apply(
    lambda x: pd.Series(parse_code_construction(x))
)

# Check groups with multiple variants in original data
sample_groups = df.groupby(['Country', 'Element', 'Renovation', 'Year_range']).size()
multi_variant_groups = sample_groups[sample_groups > 1].head(10)

for (country, element, renovation, year_range), count in multi_variant_groups.items():
    group_data = df[(df['Country'] == country) & 
                    (df['Element'] == element) & 
                    (df['Renovation'] == renovation) & 
                    (df['Year_range'] == year_range)]
    
    filtered_group = filtered_df_temp[(filtered_df_temp['Country'] == country) & 
                                     (filtered_df_temp['Element'] == element) & 
                                     (filtered_df_temp['Renovation'] == renovation) & 
                                     (filtered_df_temp['Year_range'] == year_range)]
    
    print(f"\nGroup: {country}.{element}.{renovation}.{year_range}")
    print(f"  Original variants: {len(group_data)}")
    print(f"  U values: {sorted(group_data['U'].values, reverse=True)}")
    print(f"  Max U: {group_data['U'].max()}")
    print(f"  Filtered variants: {len(filtered_group)}")
    if len(filtered_group) > 0:
        print(f"  Kept variant: {filtered_group['Number_Construction_Variant'].values[0]}")
        print(f"  Kept U value: {filtered_group['U'].values[0]}")

# Verify no group has more than one variant
group_counts = filtered_df_temp.groupby(['Country', 'Element', 'Renovation', 'Year_range']).size()
max_variants = group_counts.max()
print(f"\nMaximum variants per group in filtered data: {max_variants}")
if max_variants > 1:
    print("WARNING: Some groups still have multiple variants!")
else:
    print("✓ All groups have exactly one variant.")


Verification - Checking that each group has exactly one variant:

Group: DE.Ceiling.ReEx.04
  Original variants: 2
  U values: [np.float64(2.1), np.float64(1.6)]
  Max U: 2.1
  Filtered variants: 1
  Kept variant: 1
  Kept U value: 2.1

Group: DE.Ceiling.ReEx.07
  Original variants: 2
  U values: [np.float64(0.5), np.float64(0.5)]
  Max U: 0.5
  Filtered variants: 1
  Kept variant: 1
  Kept U value: 0.5

Group: DE.Ceiling.ReEx.08
  Original variants: 3
  U values: [np.float64(0.4), np.float64(0.3), np.float64(0.131)]
  Max U: 0.4
  Filtered variants: 1
  Kept variant: 1
  Kept U value: 0.4

Group: DE.Ceiling.ReEx.09
  Original variants: 2
  U values: [np.float64(0.35), np.float64(0.27)]
  Max U: 0.35
  Filtered variants: 1
  Kept variant: 1
  Kept U value: 0.35

Group: DE.Ceiling.ReEx.10
  Original variants: 2
  U values: [np.float64(0.3), np.float64(0.24)]
  Max U: 0.3
  Filtered variants: 1
  Kept variant: 1
  Kept U value: 0.3

Group: DE.Ceiling.ReEx.11
  Original variants: 2
  U v

In [26]:
# Save the filtered dataframe to a new CSV file
output_file = '/mnt/d/heatpump_data/u_values/germany_u_values_cleaned_filtered.csv'
filtered_df.to_csv(output_file, index=False)

print(f"Filtered CSV saved to: {output_file}")
print(f"Total rows saved: {len(filtered_df)}")

Filtered CSV saved to: /mnt/d/heatpump_data/u_values/germany_u_values_cleaned_filtered.csv
Total rows saved: 62
